# Platform Observability: Step 1 Discovery & Validation

**Purpose:** Confirm which `SNOWFLAKE.ACCOUNT_USAGE` sources exist in this account, how much history they hold, and how activity is distributed. Results from this notebook decide the scope and sizing of the `PLATFORM_OBSERVABILITY` database and its dbt models.

**Project scope**
1. History of objects created, altered, or dropped (event-based, daily)
2. History of all activity by type: DDL (create/alter/drop), DML, SELECT, grants, procedure and function calls
3. History of Cortex usage down to the individual user

**Design decisions so far**
- Snowflake Enterprise edition (so `ACCESS_HISTORY` is available)
- Transformations in dbt Cloud, incremental with a 3-day lookback
- Full SQL text retained; history purged after 2 years
- Initial backfill loads all history Snowflake still holds

**How to run**
- Import into Snowsight: *Projects > Notebooks > Import .ipynb file*
- Run with a role that has `IMPORTED PRIVILEGES` on the `SNOWFLAKE` database (for example `ACCOUNTADMIN`) and any XS warehouse
- Every query is read-only


## Q1. Which Cortex / AI usage views exist in this account?

Determines which services we can include in HIST_CORTEX_USAGE.

**What to look for:** Which Cortex and Snowflake Intelligence usage views appear. Each one becomes a candidate source for `HIST_CORTEX_USAGE`.

In [ ]:
SELECT table_name
FROM   SNOWFLAKE.INFORMATION_SCHEMA.VIEWS
WHERE  table_schema = 'ACCOUNT_USAGE'
  AND (table_name LIKE 'CORTEX%' OR table_name LIKE 'SNOWFLAKE_INTELLIGENCE%')
ORDER  BY table_name;

## Q2. Columns of those views (looking for USER_ID / USER_NAME / USERNAME,

QUERY_ID, MODEL_NAME, TOKENS, CREDITS). Tells us which services support per-user attribution.

**What to look for:** Views that carry `USER_ID`, `USER_NAME` or `USERNAME` support per-user attribution. Views without one are metered at service level only.

In [ ]:
SELECT table_name,
       column_name,
       data_type
FROM   SNOWFLAKE.INFORMATION_SCHEMA.COLUMNS
WHERE  table_schema = 'ACCOUNT_USAGE'
  AND (table_name LIKE 'CORTEX%' OR table_name LIKE 'SNOWFLAKE_INTELLIGENCE%')
ORDER  BY table_name, ordinal_position;

## Q3. Backfill depth and volume for the two big sources.

Sizes the initial load.

**What to look for:** The earliest date sets how far back the initial backfill can go (typically about 365 days).

In [ ]:
SELECT 'QUERY_HISTORY'        AS source,
       MIN(start_time)        AS earliest,
       MAX(start_time)        AS latest,
       COUNT(*)               AS total_rows
FROM   SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
UNION ALL
SELECT 'ACCESS_HISTORY',
       MIN(query_start_time),
       MAX(query_start_time),
       COUNT(*)
FROM   SNOWFLAKE.ACCOUNT_USAGE.ACCESS_HISTORY;

## Q4. Daily query volume and full-text size, last 30 days.

Since we are storing full SQL text, this estimates storage growth.

**What to look for:** Average text size and daily MB. Multiply by 730 days for a rough 2-year storage estimate before compression (Snowflake typically compresses text several-fold).

In [ ]:
SELECT DATE(start_time)                                   AS activity_date,
       COUNT(*)                                           AS queries,
       ROUND(AVG(LENGTH(query_text)))                     AS avg_text_chars,
       ROUND(SUM(LENGTH(query_text)) / POWER(1024, 2), 1) AS text_mb_uncompressed
FROM   SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE  start_time >= DATEADD(day, -30, CURRENT_TIMESTAMP())
GROUP  BY 1
ORDER  BY 1 DESC;

## Q5. Query type distribution, last 30 days.

We use this to finalize the ACTIVITY_CATEGORY mapping (DDL_CREATE / DDL_ALTER / DDL_DROP / DML / SELECT / DCL / CALL / OTHER).

**What to look for:** Every `QUERY_TYPE` listed here needs a home in the `ACTIVITY_CATEGORY` mapping. Flag any you want grouped differently.

In [ ]:
SELECT query_type,
       COUNT(*) AS queries
FROM   SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE  start_time >= DATEADD(day, -30, CURRENT_TIMESTAMP())
GROUP  BY 1
ORDER  BY 2 DESC;

## Q6. DDL events captured by ACCESS_HISTORY, last 30 days.

This is the backbone of HIST_OBJECT_CHANGES: confirms which object types and operations (CREATE / ALTER / DROP / REPLACE / UNDROP) appear.

**What to look for:** The object domains and operation types (CREATE, ALTER, DROP, REPLACE, UNDROP) that `HIST_OBJECT_CHANGES` will record.

In [ ]:
SELECT object_modified_by_ddl:"objectDomain"::STRING  AS object_domain,
       object_modified_by_ddl:"operationType"::STRING AS operation_type,
       COUNT(*)                                       AS events
FROM   SNOWFLAKE.ACCOUNT_USAGE.ACCESS_HISTORY
WHERE  query_start_time >= DATEADD(day, -30, CURRENT_TIMESTAMP())
  AND  object_modified_by_ddl IS NOT NULL
GROUP  BY 1, 2
ORDER  BY 1, 2;

## Q7. Sample DDL events (raw shape) so we can design the flattening logic.



**What to look for:** The raw JSON shape of `OBJECT_MODIFIED_BY_DDL`, which drives the flattening logic in the dbt model.

In [ ]:
SELECT query_id,
       query_start_time,
       user_name,
       object_modified_by_ddl
FROM   SNOWFLAKE.ACCOUNT_USAGE.ACCESS_HISTORY
WHERE  query_start_time >= DATEADD(day, -7, CURRENT_TIMESTAMP())
  AND  object_modified_by_ddl IS NOT NULL
LIMIT  20;

## Q8. Cross-check: daily creates and drops from the object metadata views,

last 30 days. Lets us reconcile against Q6 during validation.

**What to look for:** Daily create and drop counts by object type. We reconcile these against Q6 when validating Step 4.

In [ ]:
SELECT 'TABLE' AS object_type, DATE(created) AS d, 'CREATED' AS event, COUNT(*) AS n
FROM   SNOWFLAKE.ACCOUNT_USAGE.TABLES
WHERE  created >= DATEADD(day, -30, CURRENT_TIMESTAMP()) GROUP BY 1, 2, 3
UNION ALL
SELECT 'TABLE', DATE(deleted), 'DROPPED', COUNT(*)
FROM   SNOWFLAKE.ACCOUNT_USAGE.TABLES
WHERE  deleted >= DATEADD(day, -30, CURRENT_TIMESTAMP()) GROUP BY 1, 2, 3
UNION ALL
SELECT 'VIEW', DATE(created), 'CREATED', COUNT(*)
FROM   SNOWFLAKE.ACCOUNT_USAGE.VIEWS
WHERE  created >= DATEADD(day, -30, CURRENT_TIMESTAMP()) GROUP BY 1, 2, 3
UNION ALL
SELECT 'SCHEMA', DATE(created), 'CREATED', COUNT(*)
FROM   SNOWFLAKE.ACCOUNT_USAGE.SCHEMATA
WHERE  created >= DATEADD(day, -30, CURRENT_TIMESTAMP()) GROUP BY 1, 2, 3
UNION ALL
SELECT 'FUNCTION', DATE(created), 'CREATED', COUNT(*)
FROM   SNOWFLAKE.ACCOUNT_USAGE.FUNCTIONS
WHERE  created >= DATEADD(day, -30, CURRENT_TIMESTAMP()) GROUP BY 1, 2, 3
UNION ALL
SELECT 'PROCEDURE', DATE(created), 'CREATED', COUNT(*)
FROM   SNOWFLAKE.ACCOUNT_USAGE.PROCEDURES
WHERE  created >= DATEADD(day, -30, CURRENT_TIMESTAMP()) GROUP BY 1, 2, 3
ORDER  BY d DESC, object_type, event;

## Q9. Cortex AI Functions usage by user, last 30 days (if the view exists).

Validates per-user attribution and the USER_ID -> name join.

**What to look for:** Per-user Cortex credits. If this errors, the view is not available in this account and we fall back to older Cortex views.

In [ ]:
SELECT u.name                      AS user_name,
       c.function_name,
       c.model_name,
       COUNT(DISTINCT c.query_id)  AS queries,
       ROUND(SUM(c.credits), 4)    AS credits
FROM   SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY c
LEFT   JOIN SNOWFLAKE.ACCOUNT_USAGE.USERS u
       ON u.user_id = c.user_id
WHERE  c.start_time >= DATEADD(day, -30, CURRENT_TIMESTAMP())
GROUP  BY 1, 2, 3
ORDER  BY credits DESC;

## Next step

Share the results of Q2, Q5 and Q6 at minimum (screenshots or CSV exports). They determine:

| Result | Decides |
|---|---|
| Q1, Q2 | Which Cortex services are included and which support per-user attribution |
| Q3, Q4 | Backfill size and 2-year storage estimate |
| Q5 | Final `ACTIVITY_CATEGORY` mapping |
| Q6, Q7 | Object-change model design |
| Q8 | Reconciliation baseline for validation |
| Q9 | Per-user Cortex attribution approach |

**Step 2** then creates the `PLATFORM_OBSERVABILITY` database, schemas (`STAGING`, `HISTORY`, `MARTS`), warehouse, dbt Cloud service user and role, and grants.
